In [135]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet
import mlflow
import mlflow.prophet
import os

# Initialize local MLflow experiment
mlflow.set_experiment("SubClass_Retail_Forecast")

<Experiment: artifact_location='file:///c:/Users/jaypm/OneDrive/Documents/GitHub/advanced_business_analytics/mlruns/1', creation_time=1776619041662, experiment_id='1', last_update_time=1776619041662, lifecycle_stage='active', name='SubClass_Retail_Forecast', tags={}, trace_location=None, workspace='default'>

In [136]:
# Load data
csv_filename = csv_filename = os.path.join('data', 'Queryclassproject_Full_2026-04-02.csv') # Ensure this is in your project folder
df = pd.read_csv(csv_filename)

# Cleaning logic (updated for local pandas)
df['month_year'] = df['month'].str[-4:]
df['month_number'] = df['month'].str.extract(r'Fiscal Period (\d+)').astype(int)

# Create datetime objects
df['ds'] = pd.to_datetime('1-' + df['month_number'].astype(str) + '-' + df['month_year'], format='%d-%m-%Y')
df_clean = df.rename(columns={'sales units': 'y', 'sub class': 'sub_class'})

# Filter out March 2026 (incomplete month)
df_clean = df_clean[~((df_clean['ds'].dt.month == 3) & (df_clean['ds'].dt.year == 2026))]

# Drop unnecessary columns
df_clean = df_clean[['ds', 'y', 'sub_class']].dropna()

In [137]:
import mlflow
import mlflow.prophet
from prophet import Prophet
import numpy as np
import pandas as pd

# 1. Force end any existing run that might be stuck in memory
if mlflow.active_run():
    mlflow.end_run()

sub_classes = df_clean['sub_class'].unique()
all_forecasts = []

print(f"Starting forecast for {len(sub_classes)} sub-classes...")

for sc in sub_classes:
    # Filter and aggregate data for this specific sub-class
    group_df = df_clean[df_clean['sub_class'] == sc].groupby('ds')['y'].sum().reset_index()
    
    # Skip if there's no data to fit
    if len(group_df) < 2:
        print(f"Skipping {sc}: Not enough data points.")
        continue

    try:
        # Start the MLflow run
        with mlflow.start_run(run_name=f"SC_{sc}"):
            # Fit Prophet Model
            model = Prophet(seasonality_mode='multiplicative', yearly_seasonality=True)
            model.fit(group_df)
            
            # Log params & model to MLflow
            mlflow.log_param("sub_class", sc)
            mlflow.log_param("model_type", "baseline")
            mlflow.prophet.log_model(model, name="model")
            
            # Forecast 12 months into the future
            future = model.make_future_dataframe(periods=12, freq='MS')
            forecast = model.predict(future)
            
            # Calculate MAPE (Simple version)
            # We merge actuals (y) with predictions (yhat) to compare
            merged = group_df.merge(forecast[['ds', 'yhat']], on='ds')
            
            # Use 1e-5 to prevent division by zero errors
            mape = np.mean(np.abs((merged['y'] - merged['yhat']) / (merged['y'] + 1e-5))) * 100
            mlflow.log_metric("mape", mape)
            
            # Store results for the global dataframe
            forecast['sub_class'] = sc
            all_forecasts.append(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper', 'sub_class']])
            
            print(f"Completed: {sc} | MAPE: {mape:.2f}%")

    except Exception as e:
        print(f"Error modeling {sc}: {e}")
        # If the run crashed but stayed open, close it before moving to the next sub-class
        if mlflow.active_run():
            mlflow.end_run()

# Combine all forecasts into one final DataFrame
if all_forecasts:
    final_forecast_df = pd.concat(all_forecasts, ignore_index=True)
    print("\nAll forecasting complete and logged to MLflow.")
else:
    print("\nNo forecasts were generated. Check your data filtering.")

16:02:03 - cmdstanpy - INFO - Chain [1] start processing


Starting forecast for 13 sub-classes...


16:02:03 - cmdstanpy - INFO - Chain [1] done processing
16:02:03 - cmdstanpy - INFO - Chain [1] start processing


Completed: 026-001-005-ABS FITTINGS | MAPE: 0.95%


16:02:03 - cmdstanpy - INFO - Chain [1] done processing
16:02:04 - cmdstanpy - INFO - Chain [1] start processing


...

Completed: 026-001-033-SEWER PIPE | MAPE: 6.06%


16:02:08 - cmdstanpy - INFO - Chain [1] done processing
16:02:08 - cmdstanpy - INFO - Chain [1] start processing


Completed: 026-001-060-S/O PIPE & FITTINGS | MAPE: 281.07%


16:02:08 - cmdstanpy - INFO - Chain [1] done processing


Completed: 026-001-004-ABS PIPE | MAPE: 1506196.10%

All forecasting complete and logged to MLflow.


In [138]:
def clear_model_runs(model_type):
    """Delete all existing MLflow runs for a given model_type before re-running."""
    runs = mlflow.search_runs(filter_string=f"params.model_type = '{model_type}'")
    if runs.empty:
        return
    client = mlflow.tracking.MlflowClient()
    for run_id in runs['run_id']:
        client.delete_run(run_id)
    print(f"Cleared {len(runs)} existing '{model_type}' runs.")

In [139]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# 1. Create the UI elements
sc_dropdown = widgets.Dropdown(options=sorted(sub_classes), description='Sub-Class:')
out = widgets.Output()

clear_model_runs("baseline")

def render_dashboard(change):
    with out:
        out.clear_output()
        selected = change['new']
        
        # --- A. Retrieve Model & Metrics ---
        # Search MLflow for the specific run to get the MAPE metric we logged
        run_info = mlflow.search_runs(filter_string=f"params.sub_class = '{selected}'").iloc[0]
        mape_score = run_info['metrics.mape']
        run_id = run_info.run_id
        
        # Load the model and history
        model = mlflow.prophet.load_model(f"runs:/{run_id}/model")
        hist = df_clean[df_clean['sub_class'] == selected].groupby('ds')['y'].sum().reset_index()
        
        # Generate the forecast components
        future = model.make_future_dataframe(periods=12, freq='MS')
        forecast = model.predict(future)

        # --- B. Layout: Dashboard with Accuracy Sticker ---
        fig = plt.figure(figsize=(15, 10))
        
        # 1. Main Forecast (Top Row)
        ax1 = plt.subplot2grid((2, 2), (0, 0), colspan=2)
        ax1.plot(hist['ds'], hist['y'], 'ko', markersize=4, label='Actual Sales')
        ax1.plot(forecast['ds'], forecast['yhat'], color='#0072B2', lw=2, label='Forecast')
        ax1.fill_between(forecast['ds'], forecast['yhat_lower'], forecast['yhat_upper'], color='#0072B2', alpha=0.2)
        ax1.set_title(f"Forecast Overview: {selected}", fontsize=14, fontweight='bold')
        
        # Add the Accuracy "Sticker" (Text Box)
        # Positioned in the top left of the main chart
        accuracy_text = f"Model Accuracy (MAPE): {mape_score:.2f}%"
        props = dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='gray')
        ax1.text(0.02, 0.95, accuracy_text, transform=ax1.transAxes, fontsize=12,
                verticalalignment='top', bbox=props, fontweight='bold', color='#0072B2')
        ax1.legend(loc='upper right')

        # 2. Trendline (Bottom Left)
        ax2 = plt.subplot2grid((2, 2), (1, 0))
        ax2.plot(forecast['ds'], forecast['trend'], color='forestgreen', lw=2)
        ax2.set_title("Long-term Trend", fontsize=12)
        ax2.fill_between(forecast['ds'], forecast['trend_lower'], forecast['trend_upper'], color='forestgreen', alpha=0.2)
        ax2.grid(alpha=0.2)

        # 3. Yearly Seasonality (Bottom Right)
        ax3 = plt.subplot2grid((2, 2), (1, 1))
        if 'yearly' in forecast:
            # Show a 12-month cycle
            cycle = forecast.head(12).copy()
            ax3.plot(cycle['ds'].dt.month_name(), cycle['yearly'], marker='o', color='orange', lw=2)
            ax3.set_title("Yearly Seasonality (Unit Impact)", fontsize=12)
            plt.xticks(rotation=45)
        ax3.grid(alpha=0.2)

        plt.tight_layout()
        plt.show()
            # Calculate WAPE instead of MAPE
        sum_abs_error = np.sum(np.abs(merged['y'] - merged['yhat']))
        sum_actuals = np.sum(merged['y'])

        # WAPE avoids the division-by-zero issue at the row level
        wape = (sum_abs_error / sum_actuals) * 100 if sum_actuals > 0 else 0
        mlflow.log_metric("wape", wape)
# Connect and display
sc_dropdown.observe(render_dashboard, names='value')


Cleared 13 existing 'baseline' runs.


Okay two of the subclasses have extremely off metrics. ABS Pipe and S/o Pipe and fittings. I am going to check and see if their volume is comparable to the other sub classes. 

In [140]:
import pandas as pd

results_list = []

# Loop through all sub-classes to pull metrics from MLflow
for sc in sub_classes:
    # 1. Get MLflow run info
    run_info = mlflow.search_runs(filter_string=f"params.sub_class = '{sc}'").iloc[0]
    mape = run_info.get('metrics.mape', 0)

    
    # 2. Pull volume from the ORIGINAL dataframe 'df'
    # Note: We use 'sub class' (with a space) as it appears in your raw CSV
    try:
        raw_total_vol = df[df['sub class'] == sc]['sales units'].sum()
    except KeyError:
        # Fallback in case 'sub_class' was already renamed in 'df'
        raw_total_vol = df[df['sub_class'] == sc]['sales units'].sum()
    
    results_list.append({
        "Sub-Class": sc,
        "Raw Total Volume": raw_total_vol,
        "MAPE (%)": f"{mape:.2f}%",
               "Status": "✅ Reliable" if mape < 25 else "⚠️ High Error"
    })

# Create and display the summary
summary_df = pd.DataFrame(results_list).sort_values("Raw Total Volume", ascending=False)
print(f"Total Sub-Classes Tracked: {len(summary_df)}")
print(summary_df.to_markdown(index=False))

IndexError: single positional indexer is out-of-bounds

Okay i can probably drop the two lowest Sub classes from my analysis. 

In [ ]:
# List the exact names of the sub-classes you want to remove
to_drop = ['026-001-060-S/O PIPE & FITTINGS', '026-001-004-ABS PIPE']

# Filter the dataframe
df_clean = df_clean[~df_clean['sub_class'].isin(to_drop)]


In [ ]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np

# 1. Update list of sub-classes from your cleaned DF
available_subclasses = sorted(df_clean['sub_class'].unique())

sc_dropdown = widgets.Dropdown(options=available_subclasses, description='Sub-Class:')
out = widgets.Output()

def render_mape_dashboard(change):
    with out:
        out.clear_output()
        selected = change['new']
        
        # --- A. Retrieve Model & Data ---
        run_info = mlflow.search_runs(
    filter_string=f"params.sub_class = '{selected}' and params.model_type = 'baseline'"
).iloc[0]
        model = mlflow.prophet.load_model(f"runs:/{run_info.run_id}/model")
        
        # Use df_clean for the "Actuals" dots
        hist = df_clean[df_clean['sub_class'] == selected].groupby('ds')['y'].sum().reset_index()
        
        # Generate forecast
        future = model.make_future_dataframe(periods=12, freq='MS')
        forecast = model.predict(future)

        # --- B. Calculate MAPE Safely ---
        # Join actuals and predictions to compare
        eval_df = hist.merge(forecast[['ds', 'yhat']], on='ds')
        
        # Filter out rows where actual 'y' is 0 to avoid division by zero
        eval_df = eval_df[eval_df['y'] > 0]
        
        if not eval_df.empty:
            mape_val = np.mean(np.abs((eval_df['y'] - eval_df['yhat']) / eval_df['y'])) * 100
        else:
            mape_val = 0

        # --- C. Plotting ---
        fig = plt.figure(figsize=(15, 10))
        
        # Main Forecast Plot
        ax1 = plt.subplot2grid((2, 2), (0, 0), colspan=2)
        ax1.plot(hist['ds'], hist['y'], 'ko', markersize=4, label='Actuals')
        ax1.plot(forecast['ds'], forecast['yhat'], color='#0072B2', lw=2, label='Forecast')
        ax1.fill_between(forecast['ds'], forecast['yhat_lower'], forecast['yhat_upper'], alpha=0.2)
        
        # Label with standard MAPE
        ax1.text(0.02, 0.95, f"Model MAPE: {mape_val:.2f}%", transform=ax1.transAxes, 
                 fontsize=12, fontweight='bold', bbox=dict(facecolor='white', alpha=0.8))
        
        ax1.set_title(f"Final Analysis: {selected}", fontsize=14)
        ax1.legend()

        # Components
        ax2 = plt.subplot2grid((2, 2), (1, 0))
        ax2.plot(forecast['ds'], forecast['trend'], color='green')
        ax2.set_title("Growth Trend")

        ax3 = plt.subplot2grid((2, 2), (1, 1))
        if 'yearly' in forecast:
            ax3.plot(forecast.head(12)['ds'].dt.month_name(), forecast.head(12)['yearly'], color='orange', marker='o')
            ax3.set_title("Annual Seasonality")
            plt.xticks(rotation=45)

        plt.tight_layout()
        plt.show()

sc_dropdown.observe(render_mape_dashboard, names='value')


Now i will add a model with HOUST and COMPUTSA as regressors

In [ ]:
import os
import pandas as pd

# 1. Load HOUST CSV
csv_filename2 = os.path.join('data', 'HOUST.csv')
df_houst = pd.read_csv(csv_filename2) 
df_houst['ds'] = pd.to_datetime(df_houst['observation_date'])

# 2. Load COMPUTSA CSV
csv_filename3 = os.path.join('data', 'COMPUTSA.csv')
df_computsa = pd.read_csv(csv_filename3)
df_computsa['ds'] = pd.to_datetime(df_computsa['observation_date'])

df_houst['ds'] = df_houst['ds'].dt.to_period('M').dt.to_timestamp()
df_computsa['ds'] = df_computsa['ds'].dt.to_period('M').dt.to_timestamp()
# 3. Create Lags for both indicators
# .shift(n) moves the economic data n months into the "future" relative to sales
df_houst['HOUST_3M_LAG'] = df_houst['HOUSTSA'].shift(3)
df_houst['HOUST_6M_LAG'] = df_houst['HOUSTSA'].shift(6)

df_computsa['COMP_3M_LAG'] = df_computsa['COMPUTSA'].shift(3)
df_computsa['COMP_6M_LAG'] = df_computsa['COMPUTSA'].shift(6)

# 4. Merge External Factors together first
df_external = pd.merge(
    df_houst[['ds', 'HOUST_3M_LAG', 'HOUST_6M_LAG']], 
    df_computsa[['ds', 'COMP_3M_LAG', 'COMP_6M_LAG']], 
    on='ds', 
    how='outer'
)

# 5. Merge into your main cleaned sales data
df_combined = pd.merge(df_clean, df_external, on='ds', how='left')

# 6. Handle Missing Values
# Fill NaNs created by shifting and potential date mismatches
lag_cols = ['HOUST_3M_LAG', 'HOUST_6M_LAG', 'COMP_3M_LAG', 'COMP_6M_LAG']
if df_combined[lag_cols].isnull().any().any():
    print("Ensuring all lagged economic columns are filled...")
    for col in lag_cols:
        df_combined[col] = df_combined[col].ffill().bfill()

print("Merge complete. Ready for Multivariate Prophet Modeling.")
print(f"Features added: {lag_cols}")

print(df_houst)




Ensuring all lagged economic columns are filled...
Merge complete. Ready for Multivariate Prophet Modeling.
Features added: ['HOUST_3M_LAG', 'HOUST_6M_LAG', 'COMP_3M_LAG', 'COMP_6M_LAG']
     Unnamed: 0 observation_date  HOUSTSA         ds  HOUST_3M_LAG  \
0           660       2014-01-31    888.0 2014-01-01           NaN   
1           661       2014-02-28    944.0 2014-02-01           NaN   
2           662       2014-03-31    970.0 2014-03-01           NaN   
3           663       2014-04-30   1043.0 2014-04-01         888.0   
4           664       2014-05-31   1007.0 2014-05-01         944.0   
..          ...              ...      ...        ...           ...   
140         800       2025-09-30   1328.0 2025-09-01        1382.0   
141         801       2025-10-31   1272.0 2025-10-01        1420.0   
142         802       2025-11-30   1324.0 2025-11-01        1291.0   
143         803       2025-12-31   1387.0 2025-12-01        1328.0   
144         804       2026-01-31   1487.0 2

In [ ]:
from scipy import stats

# --- Regressor Selection per Sub-Class ---
CORR_THRESHOLD = 0.2  # Only keep regressors with |Pearson r| >= this

regressor_selection = {}

print(f"{'Sub-Class':<40} {'H_3M':>8} {'H_6M':>8} {'C_3M':>8} {'C_6M':>8}  Selected")
print("-" * 90)

for sc in df_combined['sub_class'].unique():
    # Aggregate to monthly level first — same as the model loop does
    group_df = (
        df_combined[df_combined['sub_class'] == sc]
        .groupby('ds')
        .agg(y=('y', 'sum'), **{col: (col, 'first') for col in lag_cols})
        .reset_index()
        .dropna(subset=lag_cols)
    )

    if len(group_df) < 5:
        regressor_selection[sc] = []
        print(f"{sc:<40}  (skipped — not enough data)")
        continue

    selected = []
    corr_display = ""
    for col in lag_cols:
        r, _ = stats.pearsonr(group_df[col], group_df['y'])
        flag = "✅" if abs(r) >= CORR_THRESHOLD else "  "
        corr_display += f"  {flag}{r:+.2f}"
        if abs(r) >= CORR_THRESHOLD:
            selected.append(col)

    regressor_selection[sc] = selected
    print(f"{sc:<40}{corr_display}  → {len(selected)} used")

print(f"\nCorrelation threshold: |r| ≥ {CORR_THRESHOLD}")

Sub-Class                                    H_3M     H_6M     C_3M     C_6M  Selected
------------------------------------------------------------------------------------------
026-001-005-ABS FITTINGS                    +0.11    +0.00    +0.03    +0.05  → 0 used
026-001-003-PVC FITTINGS                  ✅+0.21    +0.05    +0.04    +0.10  → 1 used
026-001-013-CPVC FITTINGS                   -0.04    +0.07    -0.16    -0.16  → 0 used
026-001-031-DWV FITTINGS                    +0.05    -0.04    +0.02    +0.13  → 0 used
026-001-012-CPVC PIPE                       +0.02    +0.01    -0.04    -0.01  → 0 used
026-001-043-PVC PRECUTS                     +0.18    +0.08    +0.13    +0.05  → 0 used
026-001-002-PVC PIPE                      ✅+0.27    +0.08    -0.02    -0.04  → 1 used
026-001-032-DWV PIPE                        +0.13    +0.02    +0.03    +0.14  → 0 used
026-001-035-DWV PRECUTS                     +0.13    +0.01    +0.13    +0.13  → 0 used
026-001-044-ABS PRECUTS                  

The HOUST metric which is housing starts correlates much more strongly than completed housing units. The 

In [ ]:
from prophet import Prophet
import mlflow
import mlflow.prophet
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt


clear_model_runs("houst3m")


if mlflow.active_run():
    mlflow.end_run()
    

all_forecasts = []

print(f"Running HOUST_3M_LAG models for {df_combined['sub_class'].nunique()} sub-classes...\n")

for sc in df_combined['sub_class'].unique():
    group_df = (
        df_combined[df_combined['sub_class'] == sc]
        .groupby('ds')
        .agg(y=('y', 'sum'), HOUST_3M_LAG=('HOUST_3M_LAG', 'first'))
        .reset_index()
        .dropna(subset=['HOUST_3M_LAG'])
    )

    if len(group_df) < 5:
        print(f"⚠️  Skipping {sc}: not enough data")
        continue

    try:
        with mlflow.start_run(run_name=f"HOUST3M_{sc}"):
            model = Prophet(seasonality_mode='multiplicative', yearly_seasonality=True)
            model.add_regressor('HOUST_3M_LAG')
            model.fit(group_df)

            future = model.make_future_dataframe(periods=12, freq='MS')
            future = pd.merge(future, df_external[['ds', 'HOUST_3M_LAG']], on='ds', how='left')
            future['HOUST_3M_LAG'] = future['HOUST_3M_LAG'].ffill().bfill()

            forecast = model.predict(future)

            merged = group_df.merge(forecast[['ds', 'yhat']], on='ds')
            merged = merged[merged['y'] > 0]
            mape = (np.abs(merged['y'] - merged['yhat']) / merged['y']).mean() * 100

            mlflow.log_param("sub_class", sc)
            mlflow.log_param("model_type", "houst3m")
            mlflow.log_param("regressor", "HOUST_3M_LAG")
            mlflow.log_metric("mape", mape)
            mlflow.prophet.log_model(model, name="model")

            forecast['sub_class'] = sc
            all_forecasts.append(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper', 'sub_class']])
            print(f"✅ {sc} | MAPE: {mape:.2f}%")

    except Exception as e:
        print(f"❌ Error on {sc}: {e}")
        if mlflow.active_run():
            mlflow.end_run()

if all_forecasts:
    final_forecast_df = pd.concat(all_forecasts, ignore_index=True)
    print(f"\nDone. {len(all_forecasts)} sub-classes modeled.")
else:
    print("No forecasts generated.")

# --- Dropdown Dashboard ---
available_subclasses = sorted(df_combined['sub_class'].unique())
sc_dropdown = widgets.Dropdown(options=available_subclasses, description='Sub-Class:')
out = widgets.Output()

def render_houst_dashboard(change):
    with out:
        out.clear_output()
        selected = change['new']

        # Load the houst3m model specifically
        run_info = mlflow.search_runs(
            filter_string=f"params.sub_class = '{selected}' and params.model_type = 'houst3m'"
        ).iloc[0]
        model = mlflow.prophet.load_model(f"runs:/{run_info.run_id}/model")

        # Actuals
        hist = (
            df_combined[df_combined['sub_class'] == selected]
            .groupby('ds')
            .agg(y=('y', 'sum'))
            .reset_index()
        )

        # Future with regressor attached
        future = model.make_future_dataframe(periods=12, freq='MS')
        future = pd.merge(future, df_external[['ds', 'HOUST_3M_LAG']], on='ds', how='left')
        future['HOUST_3M_LAG'] = future['HOUST_3M_LAG'].ffill().bfill()

        forecast = model.predict(future)

        # MAPE
        eval_df = hist.merge(forecast[['ds', 'yhat']], on='ds')
        eval_df = eval_df[eval_df['y'] > 0]
        mape_val = (np.abs(eval_df['y'] - eval_df['yhat']) / eval_df['y']).mean() * 100 if not eval_df.empty else 0

        # Plot
        fig = plt.figure(figsize=(15, 10))

        ax1 = plt.subplot2grid((2, 2), (0, 0), colspan=2)
        ax1.plot(hist['ds'], hist['y'], 'ko', markersize=4, label='Actuals')
        ax1.plot(forecast['ds'], forecast['yhat'], color='#0072B2', lw=2, label='Forecast')
        ax1.fill_between(forecast['ds'], forecast['yhat_lower'], forecast['yhat_upper'], alpha=0.2)
        ax1.text(0.02, 0.95, f"HOUST_3M Model MAPE: {mape_val:.2f}%", transform=ax1.transAxes,
                 fontsize=12, fontweight='bold', bbox=dict(facecolor='white', alpha=0.8))
        ax1.set_title(f"HOUST_3M Forecast: {selected}", fontsize=14)
        ax1.legend()

        ax2 = plt.subplot2grid((2, 2), (1, 0))
        ax2.plot(forecast['ds'], forecast['trend'], color='green')
        ax2.set_title("Growth Trend")

        ax3 = plt.subplot2grid((2, 2), (1, 1))
        if 'yearly' in forecast.columns:
            ax3.plot(forecast.head(12)['ds'].dt.month_name(), forecast.head(12)['yearly'], color='orange', marker='o')
            ax3.set_title("Annual Seasonality")
            plt.xticks(rotation=45)

        plt.tight_layout()
        plt.show()

display(sc_dropdown, out)

Running HOUST_3M_LAG models for 11 sub-classes...



15:52:41 - cmdstanpy - INFO - Chain [1] start processing
15:52:41 - cmdstanpy - INFO - Chain [1] done processing
15:52:42 - cmdstanpy - INFO - Chain [1] start processing


✅ 026-001-005-ABS FITTINGS | MAPE: 0.91%


15:52:42 - cmdstanpy - INFO - Chain [1] done processing
15:52:42 - cmdstanpy - INFO - Chain [1] start processing


✅ 026-001-003-PVC FITTINGS | MAPE: 2.08%


...

✅ 026-001-044-ABS PRECUTS | MAPE: 2.32%


15:52:46 - cmdstanpy - INFO - Chain [1] done processing


✅ 026-001-033-SEWER PIPE | MAPE: 4.73%

Done. 11 sub-classes modeled.


Dropdown(description='Sub-Class:', options=('026-001-002-PVC PIPE', '026-001-003-PVC FITTINGS', '026-001-005-A…

Output()

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np

available_subclasses = sorted(df_clean['sub_class'].unique())

sc_dropdown = widgets.Dropdown(options=available_subclasses, description='Sub-Class:')
model_toggle = widgets.ToggleButtons(options=['Baseline', 'HOUST_3M'], description='Model:')
out = widgets.Output()

def render_dashboard(change):
    with out:
        out.clear_output()
        selected = sc_dropdown.value
        model_type = 'baseline' if model_toggle.value == 'Baseline' else 'houst3m'

        # Load the right model from MLflow
        run_info = mlflow.search_runs(
            filter_string=f"params.sub_class = '{selected}' and params.model_type = '{model_type}'"
        ).iloc[0]
        model = mlflow.prophet.load_model(f"runs:/{run_info.run_id}/model")

        # Actuals
        hist = df_clean[df_clean['sub_class'] == selected].groupby('ds')['y'].sum().reset_index()

        # Future — attach regressor only for houst3m
        future = model.make_future_dataframe(periods=12, freq='MS')
        if model_type == 'houst3m':
            future = pd.merge(future, df_external[['ds', 'HOUST_3M_LAG']], on='ds', how='left')
            future['HOUST_3M_LAG'] = future['HOUST_3M_LAG'].ffill().bfill()

        forecast = model.predict(future)

        # MAPE
        eval_df = hist.merge(forecast[['ds', 'yhat']], on='ds')
        eval_df = eval_df[eval_df['y'] > 0]
        mape_val = (np.abs(eval_df['y'] - eval_df['yhat']) / eval_df['y']).mean() * 100 if not eval_df.empty else 0

        # Plot
        fig = plt.figure(figsize=(15, 10))

        ax1 = plt.subplot2grid((2, 2), (0, 0), colspan=2)
        ax1.plot(hist['ds'], hist['y'], 'ko', markersize=4, label='Actuals')
        ax1.plot(forecast['ds'], forecast['yhat'], color='#0072B2', lw=2, label='Forecast')
        ax1.fill_between(forecast['ds'], forecast['yhat_lower'], forecast['yhat_upper'], alpha=0.2)
        ax1.text(0.02, 0.95, f"{model_toggle.value} MAPE: {mape_val:.2f}%", transform=ax1.transAxes,
                 fontsize=12, fontweight='bold', bbox=dict(facecolor='white', alpha=0.8))
        ax1.set_title(f"{model_toggle.value} Forecast: {selected}", fontsize=14)
        ax1.legend()

        ax2 = plt.subplot2grid((2, 2), (1, 0))
        ax2.plot(forecast['ds'], forecast['trend'], color='green')
        ax2.set_title("Growth Trend")

        ax3 = plt.subplot2grid((2, 2), (1, 1))
        if 'yearly' in forecast.columns:
            ax3.plot(forecast.head(12)['ds'].dt.month_name(), forecast.head(12)['yearly'], color='orange', marker='o')
            ax3.set_title("Annual Seasonality")
            plt.xticks(rotation=45)

        plt.tight_layout()
        plt.show()

sc_dropdown.observe(render_dashboard, names='value')
model_toggle.observe(render_dashboard, names='value')
display(widgets.VBox([sc_dropdown, model_toggle]), out)

Output()